In [89]:
import random
from time import sleep
from selenium import webdriver
from selenium.webdriver.chrome.options import Options

# Set up Chrome
chrome_options = Options()
chrome_options.add_argument("--start-maximized")
driver = webdriver.Chrome(options=chrome_options)

def scrape_player(player_name):
    if not player_name.startswith("players/"):
        player_name = f"players/{player_name}"

    player_data = {}

    url = f'https://tankathon.com/{player_name}'
    driver.get(url)

    sleep(random.uniform(2, 5))
    
    try:
        popup = 'CybotCookiebotDialogBodyButtonDecline'
        driver.find_element(By.ID, popup).click()
    except:
        pass

    try:
        # Find name
        name_element = WebDriverWait(driver, 10).until(
           EC.presence_of_element_located((By.XPATH, '//h1[@class="page-title"]'))
        )
        player_data['name'] = name_element.text

        # Find team
        team_element = WebDriverWait(driver, 3).until(
           EC.presence_of_element_located((By.XPATH, '//div[@class="team-link-section"]'))
        )
        player_data['pre-draft_team'] = team_element.text

        # Find player info 
        info_section = driver.find_element(By.CLASS_NAME, "player-info")
        blocks = info_section.find_elements(By.CLASS_NAME, "data-block")
        for block in blocks:
            # Find school year
            if block.find_element(By.CLASS_NAME, "label").text.strip().lower() == "year":
                player_data['school_year'] = block.find_element(By.CLASS_NAME, "data").text.strip()
            # Find position
            elif block.find_element(By.CLASS_NAME, "label").text.strip().lower() == "position":
                player_data['position'] = block.find_element(By.CLASS_NAME, "data").text.strip()
            # Find height
            elif block.find_element(By.CLASS_NAME, "label").text.strip().lower() == "height":
                player_data['height'] = block.find_element(By.CLASS_NAME, "data").text.strip()
            # Find weight
            elif block.find_element(By.CLASS_NAME, "label").text.strip().lower() == "weight":
                player_data['weight'] = block.find_element(By.CLASS_NAME, "data").text.strip()
            # Find draft age
            elif block.find_element(By.CLASS_NAME, "label").text.strip().lower() == "age at draft":
                player_data['draft_age'] = block.find_element(By.CLASS_NAME, "data").text.strip()
            # Find birth date
            elif block.find_element(By.CLASS_NAME, "label").text.strip().lower() == "birthdate":
                player_data['birth_date'] = block.find_element(By.CLASS_NAME, "data").text.strip()
            # Find nation
            elif block.find_element(By.CLASS_NAME, "label").text.strip().lower() == "nation":
                player_data['nation'] = block.find_element(By.CLASS_NAME, "data").text.strip()
            # Find hometown
            elif block.find_element(By.CLASS_NAME, "label").text.strip().lower() == "hometown":
                player_data['home_town'] = block.find_element(By.CLASS_NAME, "data").text.strip()
            # Find high school
            elif block.find_element(By.CLASS_NAME, "label").text.strip().lower() == "high school":
                player_data['high_school'] = block.find_element(By.CLASS_NAME, "data").text.strip()

        # Find draft year
        draft_year = driver.find_element(By.CSS_SELECTOR, 'a.primary-hover[href*="/past_drafts/"]')
        player_data['draft_year'] = draft_year.text.strip().split()[0]

        # Find draft position
        draft_position = driver.find_element(By.CSS_SELECTOR, 'div.number a.primary-hover')
        player_data['draft_position'] = draft_position.text.strip().split()[0]

        # Find combine stats
        try:
            stat_blocks = driver.find_elements(By.CLASS_NAME, "combine-stat")
            for block in stat_blocks:
                label_elem = block.find_element(By.CLASS_NAME, "combine-stat-label")
                value_elem = block.find_element(By.CLASS_NAME, "combine-stat-value")
        
                # Get full label text, lowercased and underscored for key
                label_text = label_elem.text.strip().lower().replace(" ", "_")
                value_text = value_elem.text.strip()
        
                player_data[label_text] = value_text
                
        except Exception as e:
            print("[DEBUG] Combine stat extraction error:", e)

        # Find game stats
        try:
            stat_blocks = driver.find_elements(By.CLASS_NAME, "stat-container")
            for block in stat_blocks:
                label_elem = block.find_element(By.CLASS_NAME, "stat-label")
                value_elem = block.find_element(By.CLASS_NAME, "stat-data")
        
                # Normalize the label for the dictionary key
                label_text = label_elem.text.strip().lower().replace(" ", "_")
                value_text = value_elem.text.strip()
        
                player_data[label_text] = value_text
        
        except Exception as e:
            print("[DEBUG] Stat extraction error:", e)
                
    except:
        player_data['name'] = 'Info not found'
        
    player_data['url'] = url
    player_data['title'] = driver.title

    return player_data

# Call example:
data = scrape_player("damian-lillard")
print(data)


{'name': 'Damian Lillard', 'pre-draft_team': 'Weber State', 'school_year': 'Senior', 'position': 'PG', 'height': '6\'2.75" (6\'7.75" wingspan)', 'weight': '189 lbs', 'draft_age': '21.92 yrs', 'birth_date': 'Jul 15, 1990', 'draft_year': '2012', 'draft_position': '6th', 'max_vertical': '39.5"', 'lane_agility': '11.15', 'shuttle': '', '3/4_sprint': '3.34', 'standing_reach': '7\'11.5"', 'wingspan': '6\'7.75"', 'g': '32', 'mp': '36.0', 'fgm-fga': '7.5-16.1', 'fg%': '.467', '3pm-3pa': '3.1-7.5', '3p%': '.409', 'ftm-fta': '7.4-8.4', 'ft%': '.887', 'reb': '5.2', 'ast': '4.2', 'blk': '0.2', 'stl': '1.5', 'to': '2.4', 'pf': '2.0', 'pts': '25.5', 'true_shooting_%': '.635', 'effective_fg%': '.562', '3pa_rate': '.465', 'fta_rate': '.519', 'proj_nba_3p%': '.396', 'usg%': '33.0', 'ast/usg': '0.82', 'ast/to': '1.73', 'per': '34.0', 'ows/40': '.243', 'dws/40': '.051', 'ws/40': '.290', 'ortg': '128.4', 'drtg': '101.6', 'obpm': '9.1', 'dbpm': '0.7', 'bpm': '9.8', 'url': 'https://tankathon.com/players/dam

In [2]:
import random
import time
import csv
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# Set up Chrome
chrome_options = Options()
chrome_options.add_argument("--start-maximized")
driver = webdriver.Chrome(options=chrome_options)

def scrape_player(player_name):
    if not player_name.startswith("players/"):
        player_name = f"players/{player_name}"

    player_data = {}
    url = f'https://tankathon.com/{player_name}'
    driver.get(url)
    time.sleep(random.uniform(2, 5))
    
    try:
        # Close cookie popup if exists
        try:
            popup = 'CybotCookiebotDialogBodyButtonDecline'
            driver.find_element(By.ID, popup).click()
        except:
            pass

        # Player name
        name_element = WebDriverWait(driver, 10).until(
           EC.presence_of_element_located((By.XPATH, '//h1[@class="page-title"]'))
        )
        player_data['name'] = name_element.text

        # Pre-draft team
        try:
            team_element = WebDriverWait(driver, 3).until(
               EC.presence_of_element_located((By.XPATH, '//div[@class="team-link-section"]'))
            )
            player_data['pre-draft_team'] = team_element.text
        except:
            player_data['pre-draft_team'] = None

        # Info blocks
        try:
            info_section = driver.find_element(By.CLASS_NAME, "player-info")
            blocks = info_section.find_elements(By.CLASS_NAME, "data-block")
            for block in blocks:
                label = block.find_element(By.CLASS_NAME, "label").text.strip().lower()
                value = block.find_element(By.CLASS_NAME, "data").text.strip()
                mapping = {
                    "year": "school_year",
                    "position": "position",
                    "height": "height",
                    "weight": "weight",
                    "age at draft": "draft_age",
                    "birthdate": "birth_date",
                    "nation": "nation",
                    "hometown": "home_town",
                    "high school": "high_school"
                }
                if label in mapping:
                    player_data[mapping[label]] = value
        except:
            pass

        # Draft year
        try:
            draft_year = driver.find_element(By.CSS_SELECTOR, 'a.primary-hover[href*="/past_drafts/"]')
            player_data['draft_year'] = draft_year.text.strip().split()[0]
        except:
            player_data['draft_year'] = None

        # Draft position
        try:
            draft_position = driver.find_element(By.CSS_SELECTOR, 'div.number a.primary-hover')
            player_data['draft_position'] = draft_position.text.strip().split()[0]
        except:
            player_data['draft_position'] = None

        # Combine stats
        try:
            stat_blocks = driver.find_elements(By.CLASS_NAME, "combine-stat")
            for block in stat_blocks:
                label_elem = block.find_element(By.CLASS_NAME, "combine-stat-label")
                value_elem = block.find_element(By.CLASS_NAME, "combine-stat-value")
                label_text = label_elem.text.strip().lower().replace(" ", "_")
                player_data[label_text] = value_elem.text.strip()
        except:
            pass

        # Game stats
        try:
            stat_blocks = driver.find_elements(By.CLASS_NAME, "stat-container")
            for block in stat_blocks:
                label_elem = block.find_element(By.CLASS_NAME, "stat-label")
                value_elem = block.find_element(By.CLASS_NAME, "stat-data")
                label_text = label_elem.text.strip().lower().replace(" ", "_")
                player_data[label_text] = value_elem.text.strip()
        except:
            pass

    except Exception as e:
        print("[DEBUG] error scraping", url, e)
        player_data['name'] = 'Info not found'
        
    player_data['url'] = url
    player_data['title'] = driver.title

    return player_data


def get_players_from_draft(year):
    """Scrape a draft page and return all player URLs"""
    url = f"https://tankathon.com/past_drafts/{year}"
    driver.get(url)
    time.sleep(random.uniform(2, 4))

    # Grab all player links
    player_links = driver.find_elements(By.CSS_SELECTOR, "a.primary-hover[href*='players/']")
    player_names = [link.get_attribute("href").split("tankathon.com/")[-1] for link in player_links]

    # Remove duplicates
    return list(set(player_names))


all_players = []

for year in range(2004, 2026):  # inclusive 2004–2025
    print(f"Scraping draft {year}...")
    players = get_players_from_draft(year)
    for p in players:
        data = scrape_player(p)
        all_players.append(data)

# Collect all unique fieldnames across players
fieldnames = set()
for player in all_players:
    fieldnames.update(player.keys())
fieldnames = sorted(list(fieldnames))

# Write to CSV
with open("tankathon_players.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(all_players)

driver.quit()

Scraping draft 2004...
Scraping draft 2005...
Scraping draft 2006...
Scraping draft 2007...
Scraping draft 2008...
Scraping draft 2009...
Scraping draft 2010...
Scraping draft 2011...
Scraping draft 2012...
Scraping draft 2013...
Scraping draft 2014...
Scraping draft 2015...
Scraping draft 2016...
Scraping draft 2017...
Scraping draft 2018...
Scraping draft 2019...
Scraping draft 2020...
Scraping draft 2021...
Scraping draft 2022...
Scraping draft 2023...
Scraping draft 2024...
Scraping draft 2025...
